# Chapter 3 (leptonic) — Notebook 4: Fit m_top

**Goals**

- Fit m(ℓνb) with a Gaussian-signal + polynomial-background model.
- Extract m_top ± σ_stat and compare with Group H.

In [ ]:
%matplotlib inline
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt

from topmass import io, kinematics, selection, plotting, fitting, neutrino, pairing, weights, style
from topmass.constants import M_W, M_TOP

In [ ]:
io.setup()                                  # select release 2025e-13tev-beta
samples = io.build_samples()                # skim '3J1LMET30', https
events = io.load_process('ttbar', samples, fraction=0.3)
print('Number of events:', len(events))

In [ ]:
cuts = selection.SemilepCuts(n_jets_min=4, n_bjets_min=2)
events = events[selection.semilep_preselection(events, cuts)]
lep = kinematics.leading_lepton(events)
met = kinematics.met_vector(events)
jets = kinematics.jet_vectors(events)
is_b = events.jet_btag_quantile >= cuts.btag_quantile_min
b_jets = jets[is_b]
keep = ak.num(b_jets) >= 2
events, lep, met, b_jets = events[keep], lep[keep], met[keep], b_jets[keep][:, :2]

nu = neutrino.build_neutrino(lep, met)
b_lep, _ = pairing.assign_bjets(b_jets, lep)
m_top = ak.to_numpy((lep + nu + b_lep).mass)

counts, edges = np.histogram(m_top, bins=40, range=(120, 250))
result = fitting.fit_topmass(counts.astype(float), edges)
print(f'm_top = {result.params["mu"]:.2f} ± {result.errors["mu"]:.2f} GeV')

centres = 0.5 * (edges[:-1] + edges[1:])
plt.errorbar(centres, counts, yerr=np.sqrt(np.maximum(counts, 1)), fmt='o', markersize=3, label='reco')
xfit = np.linspace(edges[0], edges[-1], 400)
plt.plot(xfit, fitting.signal_plus_bkg(xfit, **{k: result.params[k] for k in ('n_sig','mu','sigma','c0','c1','c2')}), label='fit')
plt.axvline(172.5, color='grey', ls='--', label='generator $m_t$')
plt.xlabel(r'$m(\ell\nu b)$ [GeV]'); plt.legend()

## ✏️ Your turn 4.1 — closure

▶️ Change the match cut and re-run.

There is no truth top-mass branch, so we validate differently: keep only events whose chosen
leptonic b-jet ΔR-matches a generator-level `truth_jet` within `DR_MATCH`, then re-fit. On this
cleaner subset the fitted m_top should move toward the generator value 172.5 GeV.

> **Challenge (optional):** loosen `DR_MATCH` toward 0.6 — more events enter (with more wrong b-jets)
> and the fitted mass should drift.

In [ ]:
DR_MATCH = 0.4    # ✏️ try 0.2, 0.4, 0.6

truth = kinematics.truth_jet_vectors(events)     # 'events' is already the final selected set
dr = ak.fill_none(ak.min(b_lep.deltaR(truth), axis=1), 99.0)
good = ak.to_numpy(dr) < DR_MATCH
print(f'{int(good.sum())} / {len(good)} events have the leptonic b-jet truth-matched (ΔR < {DR_MATCH})')

counts, edges = np.histogram(m_top[good], bins=40, range=(120, 250))
result = fitting.fit_topmass(counts.astype(float), edges)
print(f'm_top (truth-matched) = {result.params["mu"]:.2f} ± {result.errors["mu"]:.2f} GeV   (target 172.5)')

## Wrap-up: report

1. Quote m_top ± stat with appropriate significant figures.
2. Discuss two leading systematic effects (jet energy scale, b-jet pairing, neutrino-root choice).
3. Compare with Group H's hadronic-side result.